In [0]:
from pyspark.sql import SparkSession

# Criando a tabela vazia para receber os logs de qualidade
spark.sql("""
CREATE TABLE IF NOT EXISTS qualidade_dados (
    particao_tabela STRING COMMENT 'Valor da partição dos dados verificados (ex: 202512)',
    nm_tabela STRING COMMENT 'Nome da tabela verificada',
    timestamp TIMESTAMP COMMENT 'Data e hora exata da verificação',
    regra STRING COMMENT 'Nome descritivo ou tipo da regra aplicada',
    valor_regra STRING COMMENT 'Resultado numérico ou texto da métrica calculada',
    passou BOOLEAN COMMENT 'Flag indicando sucesso (true) ou falha (false)',
    
    -- Coluna de Partição da tabela de qualidade (Data da Execução)
    anomesdia STRING COMMENT 'Data da execução no formato YYYYMMDD'
)
USING DELTA
PARTITIONED BY (anomesdia)
COMMENT 'Logs de execução de regras de Data Quality'
""")

In [0]:
import yaml
from datetime import datetime
from functools import reduce
from operator import and_
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

# --- CLASSE DE MONITORAMENTO
class MonitoramentoQualidade:
    def __init__(self, spark: SparkSession, nm_tabela: str, particao_tabela: str):
        self.spark = spark
        self.nm_tabela = nm_tabela
        self.particao_tabela = particao_tabela
        self.timestamp_execucao = datetime.now()
        self.anomesdia = self.timestamp_execucao.strftime("%Y%m%d")

    def registrar_resultado(self, regra: str, valor_regra: str, passou: bool):
        schema = StructType([
            StructField("particao_tabela", StringType(), True),
            StructField("nm_tabela", StringType(), True),
            StructField("timestamp", TimestampType(), True),
            StructField("regra", StringType(), True),
            StructField("valor_regra", StringType(), True),
            StructField("passou", BooleanType(), True),
            StructField("anomesdia", StringType(), True)
        ])
        
        dados = [(self.particao_tabela, self.nm_tabela, self.timestamp_execucao, 
                  regra, str(valor_regra), passou, self.anomesdia)]
        
        df_log = self.spark.createDataFrame(dados, schema=schema)
        df_log.write.format("delta").mode("append").saveAsTable("qualidade_dados")
        
        status = "✅ APROVADO" if passou else "❌ FALHOU"
        print(f"[{status}] Regra: {regra} | Valor: {valor_regra}")

    def check_count(self, df, min_esperado: int):
        qtd = df.count()
        self.registrar_resultado(f"Volume (Min {min_esperado})", str(qtd), qtd >= min_esperado)

    def check_unique(self, df, col_name: str):
        duplicatas = df.groupBy(col_name).count().filter("count > 1").count()
        self.registrar_resultado(f"Unicidade ({col_name})", f"{duplicatas} duplicados", duplicatas == 0)

    def check_datatype(self, df, col_name: str, tipo_esperado: str):
        mapa_tipos = dict(df.dtypes)
        if col_name not in mapa_tipos:
            self.registrar_resultado(f"Tipo ({col_name})", "COLUNA AUSENTE", False)
        else:
            tipo_real = mapa_tipos[col_name]
            passou = tipo_esperado.lower() in tipo_real.lower()
            self.registrar_resultado(f"Tipo ({col_name})", f"Real: {tipo_real} | Exp: {tipo_esperado}", passou)

    def check_custom_sql(self, query: str, check_lambda_str: str):
        try:
            resultado = self.spark.sql(query).collect()[0][0]
            check_fn = eval(check_lambda_str)
            passou = check_fn(resultado)
            self.registrar_resultado("Custom SQL", str(resultado), passou)
        except Exception as e:
            self.registrar_resultado("Erro SQL", str(e), False)

# --- ENGINE DE ORQUESTRAÇÃO ---

def resolver_dados_e_particao(spark, nm_tabela, colunas, valor_config):
    df = spark.table(nm_tabela)
    
    if not colunas:
        return df, "FULL_LOAD"

    if str(valor_config).upper() == "MAX":
        # Pega a última combinação de partições existente
        max_row = df.select(colunas).distinct().orderBy([F.desc(c) for c in colunas]).first()
        if not max_row:
            return df.filter("1=0"), "TABELA_VAZIA"
        
        dict_part = max_row.asDict()
        condicoes = [F.col(k) == v for k, v in dict_part.items()]
        
        df_filtrado = df.filter(reduce(and_, condicoes))
        str_particao = "|".join([f"{k}={v}" for k, v in dict_part.items()])
        return df_filtrado, str_particao
    
    return df, str(valor_config)

def rodar_monitoramento(caminho_yaml):
    spark = SparkSession.builder.getOrCreate()
    with open(caminho_yaml, 'r') as f:
        config = yaml.safe_load(f)

    for item in config['monitoramento']:
        tabela = item['nm_tabela']
        cols_p = item['colunas_particao']
        ref = item['particao_referencia']
        
        # 1. Resolve os dados e a string de log
        df_para_teste, label_particao = resolver_dados_e_particao(spark, tabela, cols_p, ref)
        
        print(f"\n--- Analisando: {tabela} [{label_particao}] ---")
        
        # 2. Instancia o monitor
        monitor = MonitoramentoQualidade(spark, tabela, label_particao)
        
        # 3. Executa as regras
        for r in item['regras']:
            tipo = r['tipo']
            p = r['parametros']
            
            if tipo == "check_count":
                monitor.check_count(df_para_teste, p['min_esperado'])
            elif tipo == "check_unique":
                monitor.check_unique(df_para_teste, p['col_name'])
            elif tipo == "check_datatype":
                monitor.check_datatype(df_para_teste, p['col_name'], p['tipo_esperado'])
            elif tipo == "check_custom_sql":
                # Passamos a string do lambda diretamente para o eval dentro da classe
                monitor.check_custom_sql(p['query'], p['condicao_esperada'])

In [0]:
# --- EXECUÇÃO ---
if __name__ == "__main__":
    rodar_monitoramento("/Workspace/Users/joaomh@protonmail.com/usp-mba-specialization-software-engineering/dq_config.yaml")